# Échelles d'analyse 

## Todo 

- Comprendre pourquoi wtpsplit ne fonctionne pas 
- Définir quelle est la meilleure manière de diviser le corpus 
- Mettre les pour et les contre de toutes les différentes unités d'analyses


- Pour faire analyse en %, nécessité de faire tourner ça sur le corpus entier (très très long)

In [ ]:
import pandas as pd
import spacy

In [ ]:
df = pd.read_csv(
    "../data/interim/df_repu_proportion.csv", low_memory=False, dtype={"ID_orateur": str}
)

In [ ]:
df["Texte_clean"] = df["Texte_clean"].fillna("") # nécessaire de remplacer les 35 NaN restantes par des chaînes vides pour faire tourner la fonction re

In [ ]:
import datetime
import locale

# Active la locale française (nécessaire pour le format)
locale.setlocale(locale.LC_TIME, "fr_FR.UTF-8")

In [ ]:
df["dateSeance_ts"] = pd.to_datetime(df["dateSeanceJour"], format="%A %d %B %Y")
df["dateSeance_day"] = df["dateSeance_ts"].dt.normalize()  

## Étape 1 : Définir l'unité d'analyse

5 options : 
- Intervention séparées par interruptions
- Interventions regroupées
- Phrase
- Agrégat de 3 phrases
- Agrégats de 3 phrases sans chevauchement


Les unités d'analyses inférieures à l'intervention ***permettent d'utiliser tout type de modèle indépendamment de la taille de leur fenêtre contextuelle + meilleur degré de précision dans le sens donné à la "République"***. Mais cela demanderait de reproduire ça au niveau du II pour l'appliquer ensuite sur tout : y appliquer le filtre république, l'analyse. 

==> L'échelle de la phrase permet que toute la phrase soit traitée par le model, mais il peut ne pas y avoir assez de contexte pour que le modèle comprenne le thème. 

==> L'échelle de l'agrégat de 3 phrases permet d'avoir un contexte, mais demande de faire attention aux répétitions (quand chevauchement) ou découpage aléatoirement qui n'a pas de sens réel (quand non-chevauchement). 

==> 

### Par phrase 

#### Spacy

In [ ]:
# Via Spacy (long mais fonctionne)

# Chargement du modèle spaCy
nlp = spacy.load('fr_dep_news_trf')

# Fonction qui renvoie simplement la liste des phrases
def tokenize_per_sentence(text):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]

    contexts = []
    for i, sentence in enumerate(sentences):
        contexts.append((i, sentence))   # (index phrase, phrase seule)
    
    return contexts


# Application au dataframe
new_data = []

for _, row in df.iterrows():
    contexts = tokenize_per_sentence(row["Texte_clean"])
    
    for sent_id, sentence in contexts:
        new_data.append({
            "id_syceron": row["id_syceron"],  # même ID que texte source
            "doc_ID": row["UID"],
            "sentence_id": sent_id,           # index de la phrase
            "phrase": sentence               # phrase seule
        })

df_agrégat_phrase = pd.DataFrame(new_data)

print(df_agrégat_phrase.head())

#### Spacy

In [ ]:
# Spacy (long mais marche)

# Chargement du modèle spaCy
nlp = spacy.load('fr_dep_news_trf')

# Fonction pour créer des blocs non-chevauchants de 3 phrases
def tokenize_non_overlapping(text, block_size=3):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]

    contexts = []
    for i in range(0, len(sentences), block_size):
        block_sentences = sentences[i:i + block_size]
        context = " ".join(block_sentences)
        contexts.append((i, context))   # i = index de la première phrase du bloc
    
    return contexts


# Application sur le dataframe
new_data = []

for _, row in df_match.iterrows():
    contexts = tokenize_non_overlapping(row["Texte_clean"])
    
    for sent_start_id, context in contexts:
        new_data.append({
            "id_syceron": row["id_syceron"],   # <-- même identifiant que le texte source
            "doc_ID": row["UID"],
            "sentence_start_id": sent_start_id,  # index de la phrase où commence le bloc
            "context": context
        })

df_agrégat_3_unique = pd.DataFrame(new_data)

print(df_agrégat_3_unique.head())


In [ ]:
df_agrégat_3_unique.to_csv(
    "../data/interim/df_agrégats_uniques.csv",
    index=False,
    # quoting=csv.QUOTE_ALL,  # not needed anymore ?
)

#### WtP

In [ ]:
from wtpsplit import WtP
from tqdm import tqdm
import time

wtp = WtP("wtp-canine-s-12l")

def tokenize_in_blocks_of_three(text):
    sentences = wtp.split(text)
    blocks = []

    for i in range(0, len(sentences), 3):   # blocs non-chevauchants
        block = " ".join(sentences[i:i+3])
        blocks.append((i, block))  # i = index de la première phrase du bloc
    
    return blocks

# Application au DF

new_data = []

start_time = time.time()   # ⏱️ début du chronomètre

for _, row in tqdm(df_match.iterrows(), total=df_match.shape[0], desc="Segmentation WtP"):
    
    blocks = tokenize_in_blocks_of_three(row["Texte_clean"])
    
    for start_id, block in blocks:
        new_data.append({
            "id_syceron": row["id_syceron"],
            "doc_ID": row["UID"],
            "sentence_start_id": start_id,
            "context": block
        })

end_time = time.time()   # fin

# Temps total formaté
elapsed = end_time - start_time
print(f"\n Temps total : {elapsed:.2f} secondes ({elapsed/60:.2f} minutes)")

new_df = pd.DataFrame(new_data)
print(new_df.head())

## Fusion avec df 

In [ ]:
df_match = pd.read_csv(
    "../data/interim/df_agrégats_3.csv",
    low_memory=False,
    dtype={
        "id_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
    },
)

In [ ]:
df = pd.read_csv(
    "../data/interim/df_regroup_repu_absolu.csv",
    low_memory=False,
    dtype={
        "id_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
    },
)

In [ ]:
df_merged = pd.merge(df, df_match, on="id_syceron", how="left").drop(columns=["doc_ID"])  # supprimer la colonne id du df_deputes

In [ ]:
df_merged

In [ ]:
df_merged.to_csv(
    "/Users/matthiaslevalet/Desktop/Projet de recherche/republique-cest-quoi/data/interim/df_agrégats_3_repu_absolu.csv",
     index=False,
)